# Predict Calorie Expenditure
Competition Link: https://www.kaggle.com/competitions/playground-series-s5e5/overview

In [103]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow import keras
from tensorflow.keras import layers

## Data Loading

In [104]:
df_train = pd.read_csv('https://raw.githubusercontent.com/machiwao/ml-development/refs/heads/main/kaggle/predict-calorie-expenditure/train.csv')
df_test = pd.read_csv('https://raw.githubusercontent.com/machiwao/ml-development/refs/heads/main/kaggle/predict-calorie-expenditure/test.csv')
df_sf = pd.read_csv('https://raw.githubusercontent.com/machiwao/ml-development/refs/heads/main/kaggle/predict-calorie-expenditure/sample_submission.csv')

In [105]:
df_train.head()

,id,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
0,0,male,36,189.0,82.0,26.0,101.0,41.0,150.0
1,1,female,64,163.0,60.0,8.0,85.0,39.7,34.0
2,2,female,51,161.0,64.0,7.0,84.0,39.8,29.0
3,3,male,20,192.0,90.0,25.0,105.0,40.7,140.0
4,4,female,38,166.0,61.0,25.0,102.0,40.6,146.0


In [106]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750000 entries, 0 to 749999
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   id          750000 non-null  int64  
 1   Sex         750000 non-null  object 
 2   Age         750000 non-null  int64  
 3   Height      750000 non-null  float64
 4   Weight      750000 non-null  float64
 5   Duration    750000 non-null  float64
 6   Heart_Rate  750000 non-null  float64
 7   Body_Temp   750000 non-null  float64
 8   Calories    750000 non-null  float64
dtypes: float64(6), int64(2), object(1)
memory usage: 51.5+ MB


In [107]:
df_train.describe()

,id,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
count,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000
mean,374999.500000,41.420404,174.697685,75.145668,15.421015,95.483995,40.036253,88.282781
std,216506.495284,15.175049,12.824496,13.982704,8.354095,9.449845,0.779875,62.395349
min,0.000000,20.000000,126.000000,36.000000,1.000000,67.000000,37.100000,1.000000
25%,187499.750000,28.000000,164.000000,63.000000,8.000000,88.000000,39.600000,34.000000
50%,374999.500000,40.000000,174.000000,74.000000,15.000000,95.000000,40.300000,77.000000
75%,562499.250000,52.000000,185.000000,87.000000,23.000000,103.000000,40.700000,136.000000
max,749999.000000,79.000000,222.000000,132.000000,30.000000,128.000000,41.500000,314.000000


In [108]:
df_train.isna().sum()

,0
id,0
Sex,0
Age,0
Height,0
Weight,0
Duration,0
Heart_Rate,0
Body_Temp,0
Calories,0


In [109]:
df_train.duplicated().sum()

np.int64(0)

## Feature Engineering

In [110]:
# One hot 'Sex' Column
df_train = pd.get_dummies(df_train, columns=['Sex'])
df_test = pd.get_dummies(df_test, columns=['Sex'])

## Exploratory Data Analysis

In [111]:
# Select numerical and categorical cols
num_cols = df_train.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df_train.select_dtypes(include=['object']).columns

In [112]:
# Remove Calories in num_cols
num_cols = num_cols.drop('Calories')

## Model Training

In [113]:
# Scale the all features
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_train[num_cols] = scaler.fit_transform(df_train[num_cols])
df_test[num_cols] = scaler.transform(df_test[num_cols])

In [114]:
X = df_train.drop(['Calories','id'], axis=1)
y = df_train['Calories']

In [115]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [116]:
# XGBoost Regressor
from xgboost import XGBRegressor

xgb_model = XGBRegressor()
xgb_model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=None, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

## Model Evaluation

In [117]:
xgb_y_pred = xgb_model.predict(X_test)
xgb_mse = mean_squared_error(y_test, xgb_y_pred)
xgb_mae = mean_absolute_error(y_test, xgb_y_pred)
xgb_r2 = r2_score(y_test, xgb_y_pred)
# Root Mean Squared Logarithmic Error
xgb_rmsle = np.sqrt(np.mean((np.log1p(xgb_y_pred) - np.log1p(y_test))**2))

In [118]:
print(f'XGBoost MSE: {xgb_mse}')
print(f'XGBoost MAE: {xgb_mae}')
print(f'XGBoost R2: {xgb_r2}')
print(f'XGBoost RMSLE: {xgb_rmsle}')

XGBoost MSE: 14.448780542113402
XGBoost MAE: 2.3534548784487446
XGBoost R2: 0.9962718747455283
XGBoost RMSLE: 0.0654180801890149


## Submission File

In [119]:
df_sf.head()

,id,Calories
0,750000,88.283
1,750001,88.283
2,750002,88.283
3,750003,88.283
4,750004,88.283


In [120]:
id = df_sf.pop('id')
df_test = df_test.drop("id", axis=1)
y_pred = xgb_model.predict(df_test)

# Reshape y_pred to be 1-dimensional
y_pred = y_pred.reshape(-1)  # or y_pred = y_pred.flatten()

# Check for negative values and replace with absolute value
y_pred = np.where(y_pred < 0, -y_pred, y_pred)


# Create a submission DataFrame
submission_df = pd.DataFrame({
    'id': id,
    'Calories': y_pred
})

# Save the submission DataFrame to a CSV file
submission_df.to_csv('submission_file.csv', index=False)
print("Submission file created: submission_file.csv")

Submission file created: submission_file.csv


In [121]:
submission_df.describe()

,id,Calories
count,250000.000000,250000.000000
mean,874999.500000,88.244667
std,72168.927986,62.293606
min,750000.000000,0.005645
25%,812499.750000,34.186145
50%,874999.500000,76.641361
75%,937499.250000,135.521545
max,999999.000000,310.958069
